# Notebook 02: Tiền xử lý và Feature Engineering

### Mục tiêu
- Xử lý missing values, outliers, duplicates
- Loại bỏ data leakage columns
- Tạo features mới (total_stays, season, lead_time_bin...)
- Thống kê trước/sau tiền xử lý

### Lý do chọn kỹ thuật
- **Missing values**: fill 0 cho children (logic: không ghi = không có), fill 'Unknown' cho country, drop company (94% missing)
- **Feature engineering**: tạo features dựa trên domain knowledge ngành khách sạn (lead_time_bin, season, room_mismatch, adr_per_person...)

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from src.data.loader import load_raw_data, load_config
from src.data.cleaner import (
    handle_missing, remove_leakage,
    remove_invalid, clean_pipeline,
)
from src.features.builder import build_features

config = load_config("../configs/params.yaml")

## 1. Load dữ liệu gốc

In [2]:
df_raw = load_raw_data("../data/raw/hotel_bookings.csv")
print(f"Shape gốc: {df_raw.shape}")
print(f"\nMissing values TRƯỚC xử lý:")
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])

[Loader] Loaded 119,390 rows × 32 columns from ../data/raw/hotel_bookings.csv
Shape gốc: (119390, 32)

Missing values TRƯỚC xử lý:
children         4
country        488
agent        16340
company     112593
dtype: int64


## 2. Xử lý Missing Values

In [3]:
df_step1 = handle_missing(df_raw)
print(f"\nShape sau handle_missing: {df_step1.shape}")
print(f"Missing values CÒN LẠI: {df_step1.isnull().sum().sum()}")
print(f"\nCột has_company đã tạo: {df_step1['has_company'].value_counts().to_dict()}")

[Cleaner] Missing values handled. Remaining NaN: 0

Shape sau handle_missing: (119390, 32)
Missing values CÒN LẠI: 0

Cột has_company đã tạo: {0: 112593, 1: 6797}


## 3. Loại bỏ Data Leakage

In [4]:
leakage_cols = config.get("leakage_columns", ["reservation_status", "reservation_status_date"])
df_step2 = remove_leakage(df_step1, leakage_cols)
print(f"Shape sau remove_leakage: {df_step2.shape}")
print(f"Columns removed: {leakage_cols}")

[Cleaner] Removed leakage columns: ['reservation_status', 'reservation_status_date']
Shape sau remove_leakage: (119390, 30)
Columns removed: ['reservation_status', 'reservation_status_date']


## 4. Loại bỏ dữ liệu không hợp lệ

In [5]:
df_step3 = remove_invalid(df_step2)
print(f"Shape sau remove_invalid: {df_step3.shape}")

[Cleaner] Removed 32424 invalid rows (32243 duplicates). Remaining: 86,966 rows
Shape sau remove_invalid: (86966, 30)


## 5. Feature Engineering

In [6]:
df_feat = build_features(df_step3, add_date=True)
print(f"\nShape sau feature engineering: {df_feat.shape}")
print(f"\nNew features:")
new_cols = set(df_feat.columns) - set(df_step3.columns)
for col in sorted(new_cols):
    print(f"  - {col}")

[Builder] === Starting feature engineering ===
[Builder] Added 11 new features. Total columns: 41

Shape sau feature engineering: (86966, 41)

New features:
  - adr_per_person
  - arrival_date
  - has_deposit
  - is_local
  - is_weekend_stay
  - lead_time_bin
  - room_mismatch
  - season
  - total_cost
  - total_guests
  - total_stays


In [7]:
# Preview new features
df_feat[["lead_time", "lead_time_bin", "total_stays", "total_guests",
         "season", "is_local", "room_mismatch", "has_deposit",
         "adr_per_person", "total_cost"]].head(10)

,lead_time,lead_time_bin,total_stays,total_guests,season,is_local,room_mismatch,has_deposit,adr_per_person,total_cost
0,342,180+_very_long,0,2,Summer,1,0,0,0.00,0.0
1,737,180+_very_long,0,2,Summer,1,0,0,0.00,0.0
2,7,0-7_last_minute,1,1,Summer,0,1,0,75.00,75.0
3,13,7-30_short,1,1,Summer,0,0,0,75.00,75.0
4,14,7-30_short,2,2,Summer,0,0,0,49.00,196.0
5,0,0-7_last_minute,2,2,Summer,1,0,0,53.50,214.0
6,9,7-30_short,2,2,Summer,1,0,0,51.50,206.0
7,85,30-90_medium,3,2,Summer,1,0,0,41.00,246.0
8,75,30-90_medium,3,2,Summer,1,0,0,52.75,316.5
9,23,7-30_short,4,2,Summer,1,0,0,61.50,492.0


## 6. Thống kê trước/sau

In [8]:
print("=" * 60)
print("THỐNG KÊ TRƯỚC / SAU TIỀN XỬ LÝ")
print("=" * 60)
header = f"{'Metric':<30} {'Trước':<15} {'Sau':<15}"
print(header)
print("-" * 60)
print(f"{'Số dòng':<30} {df_raw.shape[0]:<15,} {df_feat.shape[0]:<15,}")
print(f"{'Số cột':<30} {df_raw.shape[1]:<15} {df_feat.shape[1]:<15}")
print(f"{'Missing values':<30} {df_raw.isnull().sum().sum():<15,} {df_feat.isnull().sum().sum():<15,}")
print(f"{'Duplicates':<30} {df_raw.duplicated().sum():<15,} {df_feat.duplicated().sum():<15,}")

THỐNG KÊ TRƯỚC / SAU TIỀN XỬ LÝ
Metric                         Trước           Sau            
------------------------------------------------------------
Số dòng                        119,390         86,966         
Số cột                         32              41             
Missing values                 129,425         0              
Duplicates                     31,994          0              


## 7. Lưu dữ liệu đã xử lý

In [9]:
import os
save_path = "../" + config["data"]["processed_path"]
os.makedirs(os.path.dirname(save_path), exist_ok=True)
df_feat.to_csv(save_path, index=False)
print(f"Saved processed data to {save_path}")
print(f"Shape: {df_feat.shape}")

Saved processed data to ../data/processed/hotel_bookings_clean.csv
Shape: (86966, 41)


## Kết luận

- Xử lý 4 cột missing: children→0, country→Unknown, agent→0, company→has_company
- Loại bỏ 2 cột leakage: reservation_status, reservation_status_date
- Loại bỏ rows invalid (adr<0, 0 guests)
- Tạo 11 features mới cho mining và modeling

→ Tiếp theo: Notebook 03 - Mining & Clustering